# 🐍 Clase 8 · Órdenes y matching

> Enviar órdenes contra el libro y ver cómo se cruzan. Market, limit, IOC y FOK: cada tipo cambia el coste, la probabilidad de ejecución y el riesgo.

**Hoy construyes:** MatchingEngine: cómo se cruzan las órdenes.

### Cómo funciona este cuaderno

1. Escribe tu respuesta en la celda de código.
2. Debajo hay una **✅ comprobación plegada**: ejecútala con `Shift+Enter` para validarte (despliégala si quieres ver el `assert`).
3. ¿Atascado? Abre **💡 Ver solución**.

**Núcleo:** los primeros (en clase) · **Si vamos bien:** el resto · **Más:** el cuaderno de auxiliares.

### 1. Una market order se llena

Cruza una market buy de 0.5 contra el primer snapshot. Guarda `fills` y `filled` (suma de tamaños).

<sub>practicas: MatchingEngine + MARKET</sub>

In [ ]:
from exchange import Market, MatchingEngine, Order, Side, OrderType
book = Market.sample().step()
eng = MatchingEngine()
fills = None
filled = None

In [ ]:
# ✅ Comprobación — ejecútala (Shift+Enter). Está plegada a propósito.
assert fills is not None, '⏸ fills sigue en None: completa el ejercicio antes de validar'
assert filled is not None, '⏸ filled sigue en None: completa el ejercicio antes de validar'
assert abs(filled - 0.5) < 1e-9, 'una market siempre se llena si hay liquidez'
assert all(f.side == Side.BUY for f in fills)
print('ok  fills=%d' % len(fills))

<details>
<summary>💡 Ver solución</summary>

```python
order = Order('BTCUSDT', Side.BUY, 0.5, order_type=OrderType.MARKET)
fills = eng.process(order, book)
filled = sum(f.size for f in fills)
```

</details>

### 2. Una limit cruza solo a su precio

Envía una limit buy enorme (size=999) al `best_bid` actual. No debería cruzar nada (precio por debajo del ask). Guarda `n_fills`.

<sub>practicas: LIMIT + remanente</sub>

In [ ]:
from exchange import Market, MatchingEngine, Order, Side, OrderType
book = Market.sample().step()
eng = MatchingEngine()
bb = book.best_bid
n_fills = None

In [ ]:
# ✅ Comprobación — ejecútala (Shift+Enter). Está plegada a propósito.
assert n_fills is not None, '⏸ n_fills sigue en None: completa el ejercicio antes de validar'
assert n_fills == 0, 'comprar al best_bid no cruza el ask'
print('ok')

<details>
<summary>💡 Ver solución</summary>

```python
order = Order('BTCUSDT', Side.BUY, 999, price=bb, order_type=OrderType.LIMIT)
fills = eng.process(order, book)
n_fills = len(fills)
```

</details>

### 3. Una market crossing limit sí cruza

Envía una limit buy de 0.3 a un precio por encima del `best_ask` (best_ask + 100). Cruza. Guarda `filled`.

<sub>practicas: limit marketable</sub>

In [ ]:
from exchange import Market, MatchingEngine, Order, Side, OrderType
book = Market.sample().step()
eng = MatchingEngine()
ba = book.best_ask
filled = None

In [ ]:
# ✅ Comprobación — ejecútala (Shift+Enter). Está plegada a propósito.
assert filled is not None, '⏸ filled sigue en None: completa el ejercicio antes de validar'
assert abs(filled - 0.3) < 1e-9
print('ok')

<details>
<summary>💡 Ver solución</summary>

```python
order = Order('BTCUSDT', Side.BUY, 0.3, price=ba+100, order_type=OrderType.LIMIT)
fills = eng.process(order, book)
filled = sum(f.size for f in fills)
```

</details>

### 4. FOK: todo o nada

Envía una FOK buy de tamaño 9999 (más de lo que hay) a precio muy alto. Debe devolver 0 fills. Guarda `n_fills`.

<sub>practicas: OrderType.FOK</sub>

In [ ]:
from exchange import Market, MatchingEngine, Order, Side, OrderType
book = Market.sample().step()
eng = MatchingEngine()
ba = book.best_ask
n_fills = None

In [ ]:
# ✅ Comprobación — ejecútala (Shift+Enter). Está plegada a propósito.
assert n_fills is not None, '⏸ n_fills sigue en None: completa el ejercicio antes de validar'
assert n_fills == 0, 'FOK no se llena entera -> 0 fills'
print('ok')

<details>
<summary>💡 Ver solución</summary>

```python
order = Order('BTCUSDT', Side.BUY, 9999, price=ba+1000, order_type=OrderType.FOK)
fills = eng.process(order, book)
n_fills = len(fills)
```

</details>

### 5. Precio efectivo de una market

Cruza una market buy de 1.0 y calcula `eff_price` = nocional total / tamaño total. Debe ser >= best_ask (pagas el barrido).

<sub>practicas: vwap de los fills</sub>

In [ ]:
from exchange import Market, MatchingEngine, Order, Side, OrderType
book = Market.sample().step()
ba = book.best_ask
eng = MatchingEngine()
eff_price = None

In [ ]:
# ✅ Comprobación — ejecútala (Shift+Enter). Está plegada a propósito.
assert eff_price is not None, '⏸ eff_price sigue en None: completa el ejercicio antes de validar'
assert eff_price >= ba - 1e-6, 'una market barre niveles: precio efectivo >= best_ask'
print('ok  eff=%.2f best_ask=%.2f' % (eff_price, ba))

<details>
<summary>💡 Ver solución</summary>

```python
order = Order('BTCUSDT', Side.BUY, 1.0, order_type=OrderType.MARKET)
fills = eng.process(order, book)
eff_price = sum(f.price*f.size for f in fills) / sum(f.size for f in fills)
```

</details>

## Cierre

La forma en que envías la orden decide tu coste: cruzar ya, o esperar barato y arriesgarte a no ejecutar.

Si llegas al ejercicio 3 ya tienes el núcleo. Los siguientes y los auxiliares consolidan.

**Siguiente clase:** seguimos construyendo el motor sobre esta pieza.

## 🚀 Llévatelo a un `.py`

Un notebook va genial para explorar, pero el código de verdad vive en archivos `.py` que se ejecutan enteros de una vez. Abre **`matching_demo.py`**: es lo que acabas de construir, ordenado y de una pieza.

Ejecútalo desde una terminal:

```bash
python matching_demo.py
```

…o aquí mismo, en la siguiente celda:

In [ ]:
!python matching_demo.py

> Es la misma pieza que vive en el paquete `exchange/` — aquí, condensada en un archivo que puedes leer de una sentada.